# Classificação de E-mails Spam e Ham — Regressão Logística

Este notebook reproduz o fluxo de treinamento utilizado no projeto de classificação de e-mails, substituindo o algoritmo SVM pela **Regressão Logística**.

**Metodologia:** análise exploratória → limpeza → divisão estratificada (80% treino / 20% teste) → TF-IDF → treinamento → avaliação → testes com novos e-mails → salvamento.

Dataset esperado: `spam_ham_dataset.csv`, com as colunas `label` (ham/spam) e `text` (conteúdo do e-mail).

## 1. Configuração do ambiente e importação das bibliotecas

In [ ]:
import io
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

sns.set_theme(style="whitegrid")

## 2. Carregamento do dataset

In [ ]:
# Selecione o arquivo spam_ham_dataset.csv quando a janela de upload abrir
uploaded = files.upload()
nome_arquivo = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[nome_arquivo]))

# Remove uma possível coluna de índice salva no CSV
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]

print(f"Dataset carregado: {len(df)} registros")
display(df.head())

## 3. Conhecendo a estrutura dos dados

In [ ]:
df.info()

print("\nColunas disponíveis:")
print(df.columns.tolist())

colunas_necessarias = {"label", "text"}
colunas_ausentes = colunas_necessarias - set(df.columns)
if colunas_ausentes:
    raise ValueError(f"Colunas necessárias não encontradas: {sorted(colunas_ausentes)}")

## 4. Análise exploratória dos dados

In [ ]:
# Valores ausentes
print("Valores ausentes:")
display(df[["label", "text"]].isnull().sum())

# Remove registros sem texto ou classificação
df = df.dropna(subset=["label", "text"]).copy()
df["label"] = df["label"].astype(str).str.lower().str.strip()
df["text"] = df["text"].astype(str)

print("\nQuantidade por classe:")
display(df["label"].value_counts())
print("\nProporção por classe (%):")
display((df["label"].value_counts(normalize=True) * 100).round(2))

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="label", order=["ham", "spam"])
plt.title("Distribuição de e-mails Spam e Ham")
plt.xlabel("Classificação")
plt.ylabel("Quantidade")
plt.tight_layout()
plt.show()

## 5. Preparação dos dados

### 5.1. Verificação de duplicatas e classificações conflitantes

Antes de remover textos repetidos, verificamos se algum mesmo texto aparece com rótulos diferentes. Esses casos precisam ser investigados, pois não existe uma única classificação consistente para o mesmo conteúdo.

In [ ]:
duplicados = df[df.duplicated(subset=["text"], keep=False)]
conflitos = duplicados.groupby("text")["label"].nunique()
quantidade_conflitos = int((conflitos > 1).sum())

print("Mensagens duplicadas:", int(df.duplicated(subset=["text"]).sum()))
print("Textos com classificações conflitantes:", quantidade_conflitos)

if quantidade_conflitos > 0:
    raise ValueError(
        "Foram encontrados textos com rótulos diferentes. Investigue esses casos antes de continuar."
    )

df_modelo = df.drop_duplicates(subset=["text"], keep="first").copy()
print("Registros antes:", len(df))
print("Registros após remoção de duplicatas:", len(df_modelo))

### 5.2. Separação das variáveis

In [ ]:
# X: conteúdo dos e-mails; y: classificação esperada
X = df_modelo["text"]
y = df_modelo["label"]

print("Total de mensagens:", len(X))
print("\nDistribuição das classificações:")
print(y.value_counts())

### 5.3. Divisão entre treinamento e teste

Serão utilizados 80% dos dados para treinamento e 20% para teste. O parâmetro `stratify` preserva aproximadamente a proporção de ham e spam nos dois conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dados de treinamento:", len(X_train))
print("Dados de teste:", len(X_test))
print("\nDistribuição no treinamento:")
print(y_train.value_counts())
print("\nDistribuição no teste:")
print(y_test.value_counts())

## 6. Transformação dos textos com TF-IDF

O TF-IDF transforma os textos em vetores numéricos, atribuindo pesos às palavras conforme sua importância nos documentos. O vetorizador aprende o vocabulário somente com os dados de treinamento; depois, aplica esse mesmo vocabulário ao conjunto de teste, evitando vazamento de informação.

In [ ]:
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

palavras = tfidf.get_feature_names_out()
print("Formato dos dados de treinamento:", X_train_tfidf.shape)
print("Formato dos dados de teste:", X_test_tfidf.shape)
print("Quantidade de palavras identificadas:", len(palavras))
print("\nPrimeiras 20 palavras:")
print(palavras[:20])

In [ ]:
# Exemplo: maiores pesos TF-IDF do primeiro e-mail de treinamento
valores = X_train_tfidf[0].toarray().flatten()
pesos = pd.DataFrame({"Palavra": palavras, "Peso TF-IDF": valores})
pesos = pesos[pesos["Peso TF-IDF"] > 0].sort_values(
    by="Peso TF-IDF", ascending=False
)
display(pesos.head(10))

## 7. Treinamento do modelo de Regressão Logística

A Regressão Logística é um algoritmo de classificação. Neste projeto, ela aprenderá a estimar a probabilidade de um e-mail pertencer à classe spam ou ham a partir dos atributos numéricos gerados pelo TF-IDF.

### 7.1. Configuração do algoritmo

`max_iter=1000` define o limite de iterações do otimizador para buscar a convergência. `random_state=42` mantém a configuração reproduzível.

In [ ]:
modelo_lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

print("Modelo de Regressão Logística configurado!")

### 7.2. Treinamento e previsões

In [ ]:
modelo_lr.fit(X_train_tfidf, y_train)
print("Treinamento da Regressão Logística concluído!")

y_pred = modelo_lr.predict(X_test_tfidf)
print("Previsões realizadas!")
print("\nPrimeiras 10 previsões:")
print(y_pred[:10])

## 8. Avaliação do modelo

As métricas serão calculadas considerando **spam** como a classe positiva:
- **Acurácia:** proporção total de previsões corretas.
- **Precisão (Spam):** entre os e-mails previstos como spam, quantos realmente eram spam.
- **Recall (Spam):** entre todos os spams reais, quantos foram identificados.
- **F1-score (Spam):** média harmônica entre precisão e recall.

In [ ]:
acuracia = accuracy_score(y_test, y_pred)
precisao = precision_score(y_test, y_pred, pos_label="spam", zero_division=0)
revocacao = recall_score(y_test, y_pred, pos_label="spam", zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label="spam", zero_division=0)

print(f"Acurácia:  {acuracia:.4f}")
print(f"Precisão:  {precisao:.4f}")
print(f"Recall:    {revocacao:.4f}")
print(f"F1-score:  {f1:.4f}")
print("\nRelatório de classificação:")
print(classification_report(
    y_test,
    y_pred,
    labels=["ham", "spam"],
    target_names=["Ham", "Spam"],
    digits=4,
    zero_division=0
))

### 8.1. Matriz de confusão

In [ ]:
matriz = confusion_matrix(y_test, y_pred, labels=["ham", "spam"])

plt.figure(figsize=(7, 5))
sns.heatmap(
    matriz,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Ham", "Spam"],
    yticklabels=["Ham", "Spam"]
)
plt.title("Matriz de Confusão - Regressão Logística")
plt.xlabel("Classificação prevista")
plt.ylabel("Classificação real")
plt.tight_layout()
plt.show()

### 8.2. Gráfico das métricas

In [ ]:
metricas = pd.DataFrame({
    "Métrica": ["Acurácia", "Precisão (Spam)", "Recall (Spam)", "F1-score (Spam)"],
    "Resultado": [acuracia, precisao, revocacao, f1]
})

plt.figure(figsize=(9, 5))
sns.barplot(data=metricas, x="Métrica", y="Resultado")
plt.title("Desempenho da Regressão Logística")
plt.ylabel("Resultado")
plt.xlabel("Métrica")
plt.ylim(0, 1)
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 9. Testes com novos e-mails

As mensagens abaixo não fazem parte do conjunto de teste. Além da classe prevista, exibimos a probabilidade estimada pelo modelo para a classe prevista. Essa probabilidade é uma estimativa do classificador, não uma garantia de que a classificação esteja correta.

In [ ]:
novos_emails = [
    "Congratulations! You have won a free prize. Click here now!",
    "Hello, can we schedule a meeting for tomorrow?",
    "You have been selected to receive a free gift. Claim now!",
    "Please find attached the report from our last meeting."
]

novos_emails_tfidf = tfidf.transform(novos_emails)
previsoes_novas = modelo_lr.predict(novos_emails_tfidf)
probabilidades = modelo_lr.predict_proba(novos_emails_tfidf)
classes = list(modelo_lr.classes_)

for email, classificacao, probs in zip(novos_emails, previsoes_novas, probabilidades):
    prob_classe = probs[classes.index(classificacao)]
    print("E-mail:", email)
    print("Classificação:", classificacao.upper())
    print(f"Probabilidade estimada da classe prevista: {prob_classe:.2%}")
    print("-" * 70)

## 10. Resumo dos resultados

In [ ]:
resultados_lr = pd.DataFrame([{
    "Modelo": "Regressão Logística",
    "Acurácia": acuracia,
    "Precisão (Spam)": precisao,
    "Recall (Spam)": revocacao,
    "F1-score (Spam)": f1
}])

display(resultados_lr)

## 11. Salvamento do modelo

Salvamos o classificador e o vetorizador TF-IDF juntos. Para classificar novos e-mails futuramente, é necessário utilizar os dois objetos salvos e aplicar o mesmo pré-processamento.

In [ ]:
arquivo_modelo = "modelo_regressao_logistica.pkl"

joblib.dump(
    {
        "modelo": modelo_lr,
        "tfidf": tfidf
    },
    arquivo_modelo
)

print(f"Modelo e vetorizador salvos em: {arquivo_modelo}")
files.download(arquivo_modelo)

## Observações finais

- O desempenho real será conhecido somente após executar todas as células no Google Colab.
- Para comparar os algoritmos de forma justa, mantenha o mesmo dataset, a mesma limpeza, a mesma divisão treino/teste e as mesmas métricas usadas nos outros notebooks.
- A Regressão Logística e o SVM não são o mesmo algoritmo, embora ambos possam trabalhar com os vetores TF-IDF.
- Não compare apenas a acurácia: observe especialmente precisão, recall e F1-score da classe spam.